# 04 — Refining Pixel Size & Wavelength: The S5 Protocol

Two parameters that v2 keeps **fixed** by default:
- `pxY`, `pxZ` — detector pixel pitch (µm)
- `Wavelength` — X-ray wavelength (Å)

Both are first-class refinable parameters in the spec.  But on a
single calibrant image at single energy, refining either one
creates an **exact gauge null** with `L_sd`:

- For pixel size: `R_pred = L_sd · tan(2θ) / p_x` — only the ratio
  `L_sd/p_x` is observable.
- For wavelength: `2θ depends on λ`, and the per-ring shift induced
  by Δλ is observationally close to a uniform L_sd shift on
  small-2θ rings.

The fix is the **S5 protocol**: anchor one of the two with an
external Gaussian prior (typically `L_sd` from a survey instrument
to ±100 µm).  The framework accepts `Parameter.prior =
GaussianPrior(mean, std)` and the LM/Laplace machinery handles the
rest.

This notebook shows the recipe end-to-end on Varex CeO₂.


In [1]:
import os, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import numpy as np
import torch
from PIL import Image

from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_file
from midas_calibrate_v2.parameters.parameter import GaussianPrior
from midas_calibrate_v2.parameters.transforms import Logit
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual
from midas_calibrate_v2.loss.constraints import gaussian_prior_residual
from midas_calibrate_v2.inference.laplace import fisher_at_map

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
PARAMS = BASE / 'refined_MIDAS_params_Ceria_63keV_900mm_100x100_0p5s_aero_0.txt'
IMAGE  = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'

def load_v1():
    v1 = V1Params.from_file(PARAMS)
    if v1.RBinSize <= 0: v1.RBinSize = 0.25
    if v1.EtaBinSize <= 0: v1.EtaBinSize = 5.0
    v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
    v1.Width = max(v1.Width, 800.0)
    return v1

image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()
print('OK')


OK


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Three configurations on the same Varex image

We'll run three calibrations and compare σ(L_sd):

1. **Baseline** — px pinned, no prior.  Headline σ(L_sd) ≈ 0.71 µm.
2. **px refined, NO prior** — exposes the (L_sd, p_x) gauge null;
   σ(L_sd) inflates to the regularisation ridge floor.
3. **S5 protocol** — px refined + GaussianPrior on L_sd at 100 µm
   (typical survey precision).  σ(L_sd) recovers to ~100 µm and
   σ(p_y) becomes data-determined.

To get HONEST σ values when a prior is wired in, we use
`fisher_at_map`'s **per-row σ_r vector**: data residuals at the
empirical strain σ_r, prior residuals at unit σ.


In [2]:
def _flat(lap):
    out = []
    for n, o, s in zip(lap.refined_names, lap.refined_offsets, lap.refined_sizes):
        for k in range(s):
            out.append(f'{n}[{k}]' if s > 1 else n)
    return out

def _sigma_for(lap, name):
    flat = _flat(lap)
    sigma = lap.sigma_per_dim.detach().cpu().numpy()
    for nm, s in zip(flat, sigma):
        if nm == name:
            return float(s)
    return float('nan')


def run_scenario(label: str, refine_px: bool, lsd_prior_um: float | None):
    v1 = load_v1()
    spec = spec_from_v1_file(PARAMS)
    if lsd_prior_um is not None:
        spec.parameters['Lsd'].prior = GaussianPrior(
            mean=float(spec.parameters['Lsd'].init), std=lsd_prior_um,
        )
    if refine_px:
        for nm in ('pxY', 'pxZ'):
            p = spec.parameters[nm]; cur = float(p.init)
            p.refined = True
            p.bounds = (cur - 0.5, cur + 0.5); p.transform = Logit(*p.bounds)

    has_prior = any(isinstance(p.prior, GaussianPrior) for p in spec.parameters.values())
    n_refined = sum(p.numel for p in spec.parameters.values() if p.refined)

    t0 = time.time()
    res = autocalibrate_pv(
        v1, image, spec=spec,
        n_iter=4, half_window_px=8.0, snr_min=8.0,
        trim_mode='stratified_multfactor', trim_residual_pct=5.0,
        reuse_fits=True, lm_max_iter=300, verbose=False,
        distribution_report=False,
    )
    fits = res.fits_final
    with torch.no_grad():
        r = pseudo_strain_residual(
            fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, res.unpacked,
            rho_d=fits.rho_d, weights=fits.weights,
            ring_idx=fits.ring_idx,
            ring_d_spacing_A=fits.ring_d_spacing_A,
        )
        sigma_r_data = float(((r * r).mean()) ** 0.5)

    def res_fn(unp):
        rd = pseudo_strain_residual(
            fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, unp,
            rho_d=fits.rho_d, weights=fits.weights,
            ring_idx=fits.ring_idx,
            ring_d_spacing_A=fits.ring_d_spacing_A,
        )
        if has_prior:
            pr = gaussian_prior_residual(unp, spec)
            if pr.numel() > 0:
                rd = torch.cat([rd, pr])
        return rd
    n_data = int(fits.Y_pix.numel())
    n_total = int(res_fn(res.unpacked).numel())
    n_prior = n_total - n_data
    sigma_r_arg = (torch.cat([
        torch.full((n_data,), sigma_r_data, dtype=torch.float64),
        torch.ones(n_prior, dtype=torch.float64),
    ]) if n_prior > 0 else sigma_r_data)
    lap = fisher_at_map(spec, res_fn, res.unpacked,
                        sigma_r=sigma_r_arg, ridge=1e-9,
                        dtype=torch.float64, device='cpu')
    return dict(
        label=label,
        elapsed_s=time.time() - t0,
        n_refined=n_refined,
        Lsd=float(res.unpacked['Lsd']),
        sigma_Lsd_um=_sigma_for(lap, 'Lsd'),
        pxY=float(res.unpacked['pxY']),
        sigma_pxY_um=_sigma_for(lap, 'pxY') if refine_px else float('nan'),
    )


print('Running 3 scenarios…')
rows = [
    run_scenario('baseline (px pinned)', refine_px=False, lsd_prior_um=None),
    run_scenario('px refined, no prior', refine_px=True, lsd_prior_um=None),
    run_scenario('S5: px + σ_Lsd 100 µm', refine_px=True, lsd_prior_um=100.0),
]

print(f'\n{"scenario":<28s}  {"σ(L_sd) [µm]":>14s}  '
      f'{"σ(pxY) [µm]":>14s}  {"refined":>9s}  {"wall":>6s}')
for r in rows:
    sp = r['sigma_pxY_um']
    sp_str = f'{sp:>14.4e}' if np.isfinite(sp) else f'{"n/a":>14s}'
    print(f'  {r["label"]:<26s}  {r["sigma_Lsd_um"]:>14.4e}  '
          f'{sp_str}  {r["n_refined"]:>9d}  {r["elapsed_s"]:>6.1f}s')


Running 3 scenarios…


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in


scenario                        σ(L_sd) [µm]     σ(pxY) [µm]    refined    wall
  baseline (px pinned)            1.6065e+00             n/a         20    27.0s
  px refined, no prior            0.0000e+00      0.0000e+00         22    21.5s
  S5: px + σ_Lsd 100 µm           1.0000e+02      1.6746e-02         22    22.1s


## What you should see

| Scenario | σ(L_sd) | σ(pxY) | Interpretation |
|---|---|---|---|
| baseline (px pinned) | ~0.7 µm | n/a | Cramér-Rao at fixed px (the headline) |
| px refined, no prior | ~0.0 (ridge floor) | ~0.0 (ridge floor) | gauge null — both σ are uninformative |
| S5: σ_Lsd prior 100 µm | exactly 100 µm | ~17 nm | prior anchors L_sd; px becomes data-determined |

The S5 σ(pxY) ≈ 17 nm = 113 ppm of 150 µm is the data-determined
Cramér-Rao under the prior-anchored gauge.  This is what you
quote when you tell a downstream pipeline what the calibration
uncertainty contributes.

## Wavelength refinement — same idea

For wavelength, the recipe is identical: anchor either L_sd or
Wavelength itself with a prior matching your independent
measurement (typical Si(111) monochromator: σ_λ/λ = 1e-4).

```python
spec.parameters['Wavelength'].refined = True
spec.parameters['Wavelength'].prior = GaussianPrior(
    mean=v1.Wavelength,
    std=v1.Wavelength * 1e-4,    # 100 ppm
)
```

See `dev/paper/runners/run_wavelength_refine.py` for the full
4-scenario sweep (no prior, σ_λ/λ ∈ {1e-4, 1e-5}).

## A subtle gotcha — Laplace under-counts under priors

The Laplace approximation gives the LOCAL Hessian σ at the
prior-pulled MAP.  When the prior dominates strongly, the local
Hessian is data-dominated (small σ) but the actual joint posterior
is broader (NUTS σ).  See the §"NUTS vs Laplace" subsection of
the paper for a concrete demonstration where Laplace under-counts
by 1.5–17×.

If the σ values matter for a publication, sample the posterior
directly with `inference.hmc.hmc_run` rather than relying on
Laplace under heavy priors.  Notebook 02 covers when to switch.
